# **Data Analysis Coding Sample** <br>
## Sejuti Mannan <br> 
---






In [2]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import duckdb
from IPython.display import display

from pathlib import Path

In [3]:
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
film_metadata_path = DATA_DIR / "film_metadata.csv"
film_performance_path = DATA_DIR / "film_performance.csv"
user_activity_path = DATA_DIR / "user_activity_with_cost.csv"

## **Task 1:** Audience Engagement & Churn Risk


---
### **Part A (SQL)**
### Initial Setup

In [4]:
con = duckdb.connect()
con.execute(f"""
CREATE TABLE film_metadata AS
SELECT *
FROM read_csv_auto('{film_metadata_path}')
""")
con.execute("SELECT * FROM film_metadata LIMIT 5").fetchdf()
# con.execute("DESCRIBE film_metadata").fetchdf()

,film_id,film_name,film_length_minutes,genre
0,F101,Wonder Woman,107,Fantasy
1,F011,Star Wars: The Force Awakens,117,Adventure
2,F005,Titanic,125,Drama
3,F082,Fast Five,89,Musical
4,F098,Dune,141,Action


In [5]:
con.execute(f"""
CREATE TABLE film_performance AS
SELECT *
FROM read_csv_auto('{film_performance_path}')
""")
con.execute("SELECT * FROM film_performance LIMIT 5").fetchdf()
# con.execute("DESCRIBE film_performance").fetchdf()

,film_id,film_name,budget_usd,marketing_spend_usd,social_mentions,first_week_views,genre,avg_rating
0,F001,Avatar,13292918,6351641,7647,375507,Horror,3.28
1,F002,Avatar: The Way of Water,1493656,656878,3815,31460,Adventure,4.75
2,F003,Avengers: Endgame,314288089,45745448,44818,3967487,Action,3.94
3,F004,Avengers: Infinity War,19762189,3578005,1552,307809,Fantasy,3.66
4,F005,Titanic,23345864,9416068,9551,1387033,Drama,3.08


In [6]:
con.execute(f"""
CREATE TABLE user_activity AS
SELECT *
FROM read_csv_auto('{user_activity_path}')
""")
con.execute("SELECT * FROM user_activity LIMIT 5").fetchdf()
# con.execute("DESCRIBE user_activity").fetchdf()

,user_id,film_id,film_name,watch_minutes,watch_date,country,cost_of_ticket
0,U168,F065,The Hunger Games,13,2025-06-15,BR,10
1,U036,F034,The Hobbit: The Battle of the Five Armies,9,2025-07-27,MX,10
2,U150,F079,Fast & Furious,10,2025-08-02,IN,15
3,U036,F068,The Hunger Games: Mockingjay - Part 2,89,2025-07-10,UK,8
4,U221,F082,Fast Five,48,2025-04-14,CA,20


### SQL Query 1: User watch summary

This query computes per-user viewing metrics including total films watched, total watch time, average session length, and the most recently watched film. Note that `total_films_watched` is the number of distinct films watched by each user and not total watch events. A subquery is used to get the latest film for each user based on `watch_date` and uses `film_id` as a tiebreaker if two films are watched on the same date.

In [ ]:
con.execute("""
SELECT user_id, COUNT(DISTINCT film_id) AS total_films_watched, SUM(watch_minutes) AS total_watch_time,
  ROUND(AVG(watch_minutes)) AS avg_session_length,
  (
    SELECT film_name
    FROM user_activity u2
    WHERE u2.user_id = u1.user_id
    ORDER BY u2.watch_date DESC, u2.film_id DESC
    LIMIT 1
  ) AS last_film_watched
FROM user_activity u1
GROUP BY user_id
ORDER BY user_id
""").fetchdf()

,user_id,total_films_watched,total_watch_time,avg_session_length,last_film_watched
0,U001,6,245.0,41.0,Wonder Woman
1,U002,7,553.0,69.0,The Hunger Games: Mockingjay - Part 2
2,U003,6,534.0,59.0,Wonder Woman
3,U004,6,314.0,45.0,Deadpool 2
4,U005,3,115.0,38.0,Dune
...,...,...,...,...,...
245,U246,9,484.0,54.0,The Hunger Games: Catching Fire
246,U247,4,294.0,59.0,Titanic
247,U248,9,622.0,62.0,The Hunger Games: Mockingjay - Part 2
248,U249,3,98.0,33.0,Star Wars: The Last Jedi


### SQL Query 2: Top 5 Films
This query retrieves the top 5 films with the highest average watch completion rate.  It joins the `film_metadata` and `user_activity` tables using `film_id`, calculates the completion rate for each watch session as `watch_minutes / film_length_minutes`, averages the completion rates, and sorts the results in descending order.

In [ ]:
con.execute("""
SELECT f.film_id, f.film_name,
  ROUND(AVG(u.watch_minutes / f.film_length_minutes),2) AS avg_watch_completion_rate
FROM film_metadata f
INNER JOIN user_activity u ON f.film_id = u.film_id
GROUP BY f.film_id, f.film_name
ORDER BY avg_watch_completion_rate DESC
LIMIT 5
""").fetchdf()

,film_id,film_name,avg_watch_completion_rate
0,F031,The Lord of the Rings: The Return of the King,0.55
1,F066,The Hunger Games: Catching Fire,0.49
2,F079,Fast & Furious,0.48
3,F005,Titanic,0.48
4,F043,Captain America: Civil War,0.46


### **Part B (Python)**

### Initial Setup

In [ ]:
film_metadata_df = pd.read_csv(film_metadata_path)
display(film_metadata_df.head(5))
# film_metadata_df.info()


,film_id,film_name,film_length_minutes,genre
0,F101,Wonder Woman,107,Fantasy
1,F011,Star Wars: The Force Awakens,117,Adventure
2,F005,Titanic,125,Drama
3,F082,Fast Five,89,Musical
4,F098,Dune,141,Action


In [ ]:
film_performance_df = pd.read_csv(film_performance_path)
display(film_performance_df.head(5))
# film_performance_df.info()

,film_id,film_name,budget_usd,marketing_spend_usd,social_mentions,first_week_views,genre,avg_rating
0,F001,Avatar,13292918,6351641,7647,375507,Horror,3.28
1,F002,Avatar: The Way of Water,1493656,656878,3815,31460,Adventure,4.75
2,F003,Avengers: Endgame,314288089,45745448,44818,3967487,Action,3.94
3,F004,Avengers: Infinity War,19762189,3578005,1552,307809,Fantasy,3.66
4,F005,Titanic,23345864,9416068,9551,1387033,Drama,3.08


In [ ]:
user_activity_df = pd.read_csv(user_activity_path)
display(user_activity_df.head(5))
# user_activity_df.info()

,user_id,film_id,film_name,watch_minutes,watch_date,country,cost_of_ticket
0,U168,F065,The Hunger Games,13,2025-06-15,BR,10
1,U036,F034,The Hobbit: The Battle of the Five Armies,9,2025-07-27,MX,10
2,U150,F079,Fast & Furious,10,2025-08-02,IN,15
3,U036,F068,The Hunger Games: Mockingjay - Part 2,89,2025-07-10,UK,8
4,U221,F082,Fast Five,48,2025-04-14,CA,20


### 1. Fit model to predict churn

In [ ]:
user_activity_df["watch_date"] = pd.to_datetime(user_activity_df["watch_date"]) # convert to datetime type
latest_date = user_activity_df["watch_date"].max()
latest_date

Timestamp('2025-09-01 00:00:00')

In [ ]:
user_summary_df = (
    user_activity_df.groupby("user_id")
    .agg(
        total_watch_time=("watch_minutes", "sum"),
        num_films_watched=("film_id", "nunique"),
        avg_ticket_cost=("cost_of_ticket", "mean"),
        country=("country", lambda x: x.mode().iloc[0]),  # country refers to most frequent country code for a user's watch events
        last_watch_date=("watch_date", "max")
    )
    .reset_index()
)

In [ ]:
user_summary_df["days_since_last_watch"] = (latest_date - user_summary_df["last_watch_date"]).dt.days  # calculate how many days since last watch

user_summary_df["churn"] = (user_summary_df["days_since_last_watch"] > 30).astype(int)
display(user_summary_df.head(10))

,user_id,total_watch_time,num_films_watched,avg_ticket_cost,country,last_watch_date,days_since_last_watch,churn
0,U001,245,6,11.000000,UK,2025-08-24,8,0
1,U002,553,7,12.125000,AU,2025-08-22,10,0
2,U003,534,6,10.666667,CA,2025-08-31,1,0
3,U004,314,6,11.714286,UK,2025-07-21,42,1
4,U005,115,3,12.666667,IN,2025-07-08,55,1
5,U006,145,3,9.333333,AU,2025-07-31,32,1
6,U007,338,5,12.600000,AU,2025-08-05,27,0
7,U008,136,3,15.666667,IN,2025-07-29,34,1
8,U009,65,3,10.000000,BR,2025-07-14,49,1
9,U010,338,6,13.285714,FR,2025-07-12,51,1


In [ ]:
X = user_summary_df[["total_watch_time", "num_films_watched", "avg_ticket_cost", "country"]]
y = user_summary_df["churn"]
numeric_features = ["total_watch_time", "num_films_watched", "avg_ticket_cost"]
categorical_features = ["country"]

In [ ]:
numeric_transformer = Pipeline(steps=[("scaler", StandardScaler())])  # standardize numeric features so one feature does not dominate
categorical_transformer = Pipeline(steps=[("onehot", OneHotEncoder(handle_unknown="ignore"))])  # convert countries into binary columns

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [ ]:
log_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42,
    stratify=y  # make sure test set has similar ratio of not churn to churn
)

In [ ]:
log_model.fit(X_train, y_train)
y_pred = log_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

Accuracy: 0.64
Precision: 0.5333333333333333
Recall: 0.2857142857142857
F1: 0.37209302325581395


I used logistic regression because it is used for binary classification and churn has two outcomes—churn or success.

### 2. Are films with higher average ticket cost correlated with lower churn rates?

To find if higher average ticket costs are correlated with lower churn rates, I created a seperate df grouped by film to find average ticket cost and churn rate. The correlation is 0.119 and positive so higher ticket costs are slightly associated with higher churn rates, meaning higher average ticket costs are not correlated with lower churn rates.

In [ ]:
user_activity_with_churn_df = user_activity_df.merge(user_summary_df[["user_id", "churn"]],
                                                     on="user_id",
                                                     how="left")
film_churn_df = user_activity_with_churn_df.groupby(["film_id"]).agg(
    avg_ticket_cost=("cost_of_ticket", "mean"),
    churn_rate=("churn", "mean")).reset_index()
display(film_churn_df.head(5))
corr = film_churn_df["avg_ticket_cost"].corr(film_churn_df["churn_rate"])
print("Correlation:", corr)

,film_id,avg_ticket_cost,churn_rate
0,F001,11.792208,0.324675
1,F005,12.333333,0.333333
2,F011,12.057143,0.314286
3,F012,11.459459,0.189189
4,F019,11.597561,0.317073


Correlation: 0.11900695544561982


### 3. Identify the top 3 predictors for churn or success, and explain why they might matter from a business perspective

The top 3 predictors for churn or success are the number of films watched, the most frequent country that the user watches from, and total_watch_time. The number of films watched and total watch time reflecting churn is important because it shows how engaged users are. However, I think the country features are more important because they indicate how consumer behavior varies across different geographical markets and business logic adjustments can be made to target regions with different preferences.

In [ ]:
# model is currently transformed so need to get one-hot encoder
onehot = log_model.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
# get encoded country column names
encoded_cat_names = onehot.get_feature_names_out(categorical_features)

feature_names = list(numeric_features) + list(encoded_cat_names)
coefficients = log_model.named_steps["model"].coef_[0]

coeff_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

coef_df = coeff_df.sort_values("coefficient", key=abs, ascending=False)
print(coeff_df)

              feature  coefficient
0    total_watch_time    -0.102440
1   num_films_watched    -1.056742
2     avg_ticket_cost     0.049531
3          country_AU    -0.110639
4          country_BR     0.000545
5          country_CA    -0.388354
6          country_DE     0.305664
7          country_FR     1.178047
8          country_IN     0.884143
9          country_JP    -0.894937
10         country_MX     0.357471
11         country_UK    -0.695406
12         country_US    -0.644011


## **Task 2:** Film Performance Prediction


---
### **Part A (SQL)**
### 1. Top 5 films with highest ROI
I calculated the cost per view to show return on investment. ROI is traditionally defined as monetary return on investment with ```net profit - total cost```. However, in this case the total profit cannot be found because the ticket cost varies per watch event so there is no definitive way to find out how much money is being generated by each view from the ```film_performance``` table during first week views. Hence, I decided to use cost per view to show how investment translates to viewership.


In [ ]:
con.execute("""
SELECT genre, SUM(budget_usd + marketing_spend_usd) / SUM(first_week_views) AS cost_per_view
FROM film_performance
GROUP BY genre
ORDER BY cost_per_view ASC
LIMIT 5;
""").fetchdf()

,genre,cost_per_view
0,Family,20.552496
1,Horror,29.992462
2,Sci-Fi,31.246457
3,Adventure,35.148144
4,Comedy,35.307840


### 2. Successful film model
I also used logistic regression here because this is another binary classification problem and logistic regression is easier to interpret compared to other models. For example, I was considering random forest classifier since it is better at capturing complex relationships but there needs to be more in-depth analysis to understand where its predicitions are coming from.

The accuracy was 0.78. Most important predictors are budget and marketing spend over genres. This makes sense because more promotion and films with more resources to spend toward actors or technology tend to do have larger first week turnouts.

In [ ]:
film_success_df = film_performance_df
film_success_df['successful_film'] = (film_success_df['first_week_views'] > 100000).astype(int)
film_success_df.head(5)

,film_id,film_name,budget_usd,marketing_spend_usd,social_mentions,first_week_views,genre,avg_rating,successful_film
0,F001,Avatar,13292918,6351641,7647,375507,Horror,3.28,1
1,F002,Avatar: The Way of Water,1493656,656878,3815,31460,Adventure,4.75,0
2,F003,Avengers: Endgame,314288089,45745448,44818,3967487,Action,3.94,1
3,F004,Avengers: Infinity War,19762189,3578005,1552,307809,Fantasy,3.66,1
4,F005,Titanic,23345864,9416068,9551,1387033,Drama,3.08,1


In [ ]:
perf_features = ['budget_usd', 'marketing_spend_usd', 'social_mentions','avg_rating','genre']
X = film_success_df[perf_features]
y = film_success_df['successful_film']
perf_numeric_features = ['budget_usd', 'marketing_spend_usd', 'social_mentions', 'avg_rating']
perf_categorical_features = ['genre']

numeric_transformer = Pipeline(steps=[("scaler", StandardScaler())])  # standardize numeric features so one feature does not dominate
categorical_transformer = Pipeline(steps=[("onehot", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, perf_numeric_features),
    ("cat", categorical_transformer, perf_categorical_features)
])

perf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y  # preserves ratio in train and test data for successful and non successful films
)

perf_model.fit(X_train, y_train)
y_pred = perf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))


Accuracy: 0.78125
Precision: 0.78125
Recall: 1.0
F1: 0.8771929824561403


In [ ]:
feature_names = perf_model.named_steps["preprocessor"].get_feature_names_out()
coefficients = perf_model.named_steps["model"].coef_[0]

importance_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "abs_coefficient": abs(coefficients)
}).sort_values("abs_coefficient", ascending=False)
importance_df.head(5)

,feature,coefficient,abs_coefficient
0,num__budget_usd,1.315730,1.315730
1,num__marketing_spend_usd,0.879597,0.879597
2,num__social_mentions,0.738452,0.738452
7,cat__genre_Comedy,-0.604409,0.604409
14,cat__genre_Thriller,0.497899,0.497899


In [ ]:
con.close()  # to prevent memory leaks in temporary database used during SQL queries

## **Task 3:** Other Questions


---
### 1. If you had these datasets and one more week, what other questions would you try to answer? How would you go about doing it, and what other resources/predictors would you want access to?

First, I would try to answer more concretely how the numerical value of first week views varies with the other columns of the ```film_performance``` table rather than just a binary classification of "success" or "not". This will help better understand which factors (ex. marketing spend vs total budget) are having a larger impact on driving first week viewership. Another question I would delve into is how different georgraphical markets and watch date from the ```user_activity``` table impact film performance. For example, there may be film genres that have a stronger appeal in certain regions or certain months. These trends can be used to better tailor audience discovery algorithms for different regions. Some other resources I would want access to is the breakdown of first week viewership by geographical location, release date of each film, more specific categorization/descriptions of films, and more information about the user during each watch event. This would help with better understand the variation in performance and the demographic types for different films.


### 2. You notice a sudden spike in watches for a particular film. How would you determine whether it’s driven by marketing, social trends, or a small group of highly active users?
The first step would be to gather more data about each of these scenarios because they are hard to attribute from the current datasets. Next, I would find when exactly this spike happened and if it correlates with a business tactic being employed by our company or the film producers (ex. marketing campaign or new platform algorithm usage). Finally, I would map where exactly the spike in watches is coming from in terms of geographical location and users. This would hint at a recent social media trend or a small group of highly active users.